## Experiment 2:

In [ ]:
from src.utils import *
import pandas as pd
import random
from src.prediction import codenet_data_analysis
import pickle

train_df =pd.read_csv('codenet/train_1725993.csv')
test_df = pd.read_csv('codenet/test_1824.csv')

all_tasks = train_df['task'].value_counts().nlargest(30).index.tolist()
all_langs = train_df['language'].value_counts().nlargest(30).index.tolist()
# num_task = [i for i in range (10,51,10)]
num_task = [5, 10]

chosen_dic = dict.fromkeys(num_task)
for num in num_task:
    chosen_dic[num] = {'chosen_lang':[], 'chosen_task':[]}


for i in num_task:
    for j in range(5):
        chosen_task = random.choices(all_tasks, k = i)
        chosen_lang = random.choices(all_langs, k = i)
        # print(f"Tasks:{chosen_task}")
        # print(f"Languages:{chosen_lang}")
        chosen_dic[i]['chosen_lang'].append(chosen_lang)
        chosen_dic[i]['chosen_task'].append(chosen_task)

        results = codenet_data_analysis(chosen_task, chosen_lang, train_df, test_df)
        print(results)
        print('========================================================')

## Log Parser

In [1]:
import re
import pandas as pd
from collections import defaultdict

def summarize_log(log_path: str, max_rounds: int = 10) -> pd.DataFrame:
    # Model line: Model: bert; Classifier: KNN
    pattern_model = re.compile(r"Model:\s*([\w.\-]+); Classifier:\s*([\w.\-]+)")

    # Final accuracy line: Final Accuracy -> Language: 0.9667, Task: 0.8000
    pattern_final = re.compile(
        r"Final Accuracy -> Language:\s*([\d.]+), Task:\s*([\d.]+)"
    )

    results = []
    current_model = None
    current_classifier = None
    round_counters = defaultdict(int)  # per (model, classifier)

    with open(log_path, "r") as f:
        for line in f:
            # Match model+classifier
            model_match = pattern_model.search(line)
            if model_match:
                current_model, current_classifier = model_match.groups()

            # Match final accuracy
            final_match = pattern_final.search(line)
            if final_match and current_model and current_classifier:
                lang_acc, task_acc = map(float, final_match.groups())
                key = (current_model, current_classifier)
                round_counters[key] += 1
                round_idx = round_counters[key]

                results.append({
                    "Model": current_model,
                    "Classifier": current_classifier,
                    "Round": round_idx,
                    "Language": lang_acc,
                    "Task": task_acc,
                })

    df = pd.DataFrame(results)

    # Pivot into wide format
    df_pivot = df.pivot_table(
        index=["Model", "Classifier"],
        columns="Round",
        values=["Language", "Task"]
    )

    # Ensure columns are ordered by round
    ordered_cols = []
    for r in range(1, max_rounds + 1):
        for metric in ["Language", "Task"]:
            col = (metric, r)
            if col in df_pivot.columns:
                ordered_cols.append(col)

    df_pivot = df_pivot[ordered_cols]

    # Flatten MultiIndex to "RoundX_Metric"
    df_pivot.columns = [f"Round{r}_{m}" for m, r in df_pivot.columns]

    return df_pivot.reset_index()


In [3]:
summarize_df = summarize_log("log/random5.log", max_rounds = 10)
summarize_df.to_csv("log/random5.csv")

In [4]:
summarize_df

,Model,Classifier,Round1_Language,Round1_Task,Round2_Language,Round2_Task,Round3_Language,Round3_Task,Round4_Language,Round4_Task,...,Round6_Language,Round6_Task,Round7_Language,Round7_Task,Round8_Language,Round8_Task,Round9_Language,Round9_Task,Round10_Language,Round10_Task
0,bert,KNN,0.8750,1.0000,0.9286,0.8571,1.0000,0.8800,0.7692,0.8462,...,0.8571,0.9048,0.7895,0.8947,0.9333,0.7333,1.0000,0.7000,0.9000,0.7500
1,bert,SVM,1.0000,1.0000,1.0000,0.9231,1.0000,1.0000,1.0000,1.0000,...,1.0000,1.0000,1.0000,0.9500,1.0000,1.0000,1.0000,0.8824,1.0000,1.0000
2,falcon11b,KNN,1.0000,0.8125,1.0000,0.8571,1.0000,0.7600,0.8462,0.6923,...,1.0000,0.8571,0.7895,0.8947,0.9333,0.7333,1.0000,0.8500,1.0000,0.8000
3,falcon11b,SVM,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,0.9524,...,1.0000,1.0000,1.0000,0.9000,1.0000,0.9500,1.0000,0.8824,1.0000,1.0000
4,falcon40b,KNN,1.0000,0.9286,1.0000,0.8000,0.8462,0.6923,0.9231,0.8462,...,0.7895,0.7368,0.9333,0.8000,1.0000,0.8500,1.0000,0.7500,1.0000,0.7647
5,falcon40b,SVM,1.0000,0.9600,1.0000,0.9231,1.0000,1.0000,1.0000,0.9524,...,1.0000,1.0000,1.0000,0.9000,1.0000,0.9500,1.0000,0.8824,1.0000,0.9412
6,falcon7b,KNN,1.0000,0.8125,1.0000,0.8571,0.9200,0.7200,0.8462,0.7692,...,0.9524,0.7619,0.7368,0.6842,0.9333,0.8000,1.0000,0.8000,0.9500,0.8000
7,falcon7b,SVM,1.0000,0.9600,0.8462,0.9231,1.0000,0.8462,1.0000,0.9048,...,0.9333,0.9333,1.0000,0.9000,1.0000,0.9000,1.0000,0.8235,1.0000,0.8235
8,gpt,KNN,0.8125,0.5000,0.5714,0.3571,0.8400,0.4800,0.3846,0.6923,...,0.5238,0.5714,0.4211,0.5789,0.8000,0.7333,0.8500,0.5500,0.8000,0.5500
9,gpt,SVM,1.0000,0.7600,0.7692,0.8462,0.8462,0.9231,0.9524,0.9048,...,0.9333,0.8667,1.0000,0.6000,0.9500,0.8000,0.9412,0.5294,0.9412,0.8824
